In [14]:
# 迭代数组
'''
NumPy 迭代器对象 numpy.nditer 提供了一种灵活访问一个或者多个数组元素的方式。
迭代器最基本的任务的可以完成对数组元素的访问。

np.nditer(数组)：创建多维迭代器，按数组内存存储顺序逐个遍历所有元素，不管数组是几维，都扁平化依次取出每个值。
默认遍历顺序：C 顺序（行优先，先走完一整行再下一行）
'''
# 示例   使用 arange() 函数创建一个 2X3 数组，并使用 nditer 对它进行迭代
import numpy as np

a = np.arange(6).reshape(2,3)
print('原始数组为：\n',a)
'''
原始数组为：
 [[0 1 2]
 [3 4 5]]
'''

print('迭代输出元素：')
for x in np.nditer(a):
    print(x,end = ', ')
print('\n')
'''
迭代输出元素：
0, 1, 2, 3, 4, 5, 

按顺序遍历每个元素
'''

for x in np.nditer(a.T):
    print(x,end = ', ')  # 0, 1, 2, 3, 4, 5,   将 a 进行转置后的顺序
print('\n')

for x in np.nditer(a.T.copy(order = 'C')):
    print(x,end = ', ')  #  0, 3, 1, 4, 2, 5, 
print('\n')

'''
.copy() 生成一块全新内存，强制复制一份新数组，order='C' 让新副本按转置后的逻辑行优先重新排列内存。
.T 是 NumPy 数组的 '转置' 属性，作用：交换数组的行和列。

for x in np.nditer(a, order='F'):Fortran order，即是列序优先；
for x in np.nditer(a.T, order='C'):C order，即是行序优先；
实例
'''


# 通过显式设置，来强制 nditer 对象使用某种顺序
a = np.arange(0,60,5).reshape(3,4)
print('原始数组为：\n',a)
'''
原始数组为：
 [[ 0  5 10 15]
 [20 25 30 35]
 [40 45 50 55]]
'''
print('以 F 的风格顺序排序：')
for x in np.nditer(a,order = 'F'):
    print(x,end = '  ')
'''
以 F 的风格顺序排序：
0  20  40  5  25  45  10  30  50  15  35  55  
'''


# 修改数组中元素的值
'''
nditer 对象有另一个可选参数 op_flags。
默认情况下，nditer 将视待迭代遍历的数组为只读对象（read-only），为了在遍历数组的同时，实现对数组元素值的修改，必须指定 readwrite 或者 writeonly 的模式。
'''
# 示例
a = np.arange(0,60,5).reshape(3,4)
print('原始数组为：\n',a)
'''
原始数组为：
 [[ 0  5 10 15]
 [20 25 30 35]
 [40 45 50 55]]
'''
for x in np.nditer(a,op_flags = ['readwrite']):
    x[...] = 2*x
print('修改后的数字是：\n',a)
'''
修改后的数字是：
 [[  0  10  20  30]
 [ 40  50  60  70]
 [ 80  90 100 110]]

op_flags=['readwrite']：
默认迭代器只能读取元素，加这个参数后，迭代变量 x 支持原地修改原数组

x[...] = 2*x
x[...] 代表当前迭代到的数组元素切片，等价把当前元素乘以 2，写回原数组对应位置
多维数组中：... 代表剩余所有维度，等价连续多个 :；
'''


# 使用外部循环
'''
nditer 类的构造器拥有 flags 参数，它可以接受下列值：
参数			描述
c_index	        可以跟踪 C 顺序的索引
f_index	        可以跟踪 Fortran 顺序的索引
multi_index	    每次迭代可以跟踪一种索引类型
external_loop	给出的值是具有多个值的一维数组，而不是零维数组
'''
#示例    迭代器遍历对应于每列，并组合为一维数组。

a = np.arange(0,60,5).reshape(3,4)
print('原始数组为：\n',a)
'''
原始数组为：
 [[ 0  5 10 15]
 [20 25 30 35]
 [40 45 50 55]]
'''

print ('修改后的数组是：')
for x in np.nditer(a,flags = ['external_loop'],order = 'F'):
    print(x,end = '  ')
'''
修改后的数组是：
[ 0 20 40]  [ 5 25 45]  [10 30 50]  [15 35 55] 
'''

print('\n')

# 广播迭代
'''
如果两个数组是可广播的，nditer 组合对象能够同时迭代它们。 
假设数组 a 的维度为 3X4，数组 b 的维度为 1X4 ，则使用以下迭代器（数组 b 被广播到 a 的大小）。
'''
a = np.arange(0,60,5).reshape(3,4)
print  ('第一个数组为：\n',a)
'''
第一个数组为：
 [[ 0  5 10 15]
 [20 25 30 35]
 [40 45 50 55]]
'''

b = np.array([1,2,3,4],dtype = int)
print('第二个数组为：\n',b)
'''
第二个数组为：
 [1 2 3 4]
'''

print ('修改后的数组为：')
for x,y in np.nditer([a,b]):
    print(f"{x}:{y}",end = '  ')
'''
修改后的数组为：
0:1  5:2  10:3  15:4  20:1  25:2  30:3  35:4  40:1  45:2  50:3  55:4  

np.nditer([数组1,数组2]) 会广播匹配两个数组形状，同步成对取出元素 (x,y)
a 形状 (3,4)，b 形状 (4,)，广播后 b 自动拉伸为 (3,4)，每行都是 [1,2,3,4]
默认按C 行优先遍历，逐行配对取值
'''

原始数组为：
 [[0 1 2]
 [3 4 5]]
迭代输出元素：
0, 1, 2, 3, 4, 5, 

0, 1, 2, 3, 4, 5, 

0, 3, 1, 4, 2, 5, 

原始数组为：
 [[ 0  5 10 15]
 [20 25 30 35]
 [40 45 50 55]]
以 F 的风格顺序排序：
0  20  40  5  25  45  10  30  50  15  35  55  原始数组为：
 [[ 0  5 10 15]
 [20 25 30 35]
 [40 45 50 55]]
修改后的数字是：
 [[  0  10  20  30]
 [ 40  50  60  70]
 [ 80  90 100 110]]
原始数组为：
 [[ 0  5 10 15]
 [20 25 30 35]
 [40 45 50 55]]
修改后的数组是：
[ 0 20 40]  [ 5 25 45]  [10 30 50]  [15 35 55]  

第一个数组为：
 [[ 0  5 10 15]
 [20 25 30 35]
 [40 45 50 55]]
第二个数组为：
 [1 2 3 4]
修改后的数组为：
0:1  5:2  10:3  15:4  20:1  25:2  30:3  35:4  40:1  45:2  50:3  55:4  

'\n修改后的数组为：\n0:1  5:2  10:3  15:4  20:1  25:2  30:3  35:4  40:1  45:2  50:3  55:4  \n\nnp.nditer([数组1,数组2]) 会广播匹配两个数组形状，同步成对取出元素 (x,y)\na 形状 (3,4)，b 形状 (4,)，广播后 b 自动拉伸为 (3,4)，每行都是 [1,2,3,4]\n默认按C 行优先遍历，逐行配对取值\n'